# Myntra Fashion Classifier

Upgrade from Fashion-MNIST (28×28 grayscale, 10 toy classes) to the **Myntra Fashion Product Images** dataset:
- 44,000 real RGB product photos
- 224×224 resolution
- Rich label hierarchy: masterCategory → subCategory → articleType
- Colour, season, gender metadata

We train and compare:
1. **MobileNetV2** — fast, mobile-friendly transfer learning
2. **EfficientNetV2-S** — higher accuracy, same training approach

And keep the full production pipeline: augmentation, multi-label heads, hierarchical fallback, evaluation, and export.

## 1. Setup

In [1]:
import matplotlib
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, applications
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pathlib, json, os

SEED     = 42
IMG_SIZE = 96       # 96×96 — 3× better than Fashion-MNIST's 28×28, trains fast on CPU
BATCH    = 64
SUBSET   = 10000    # use 10K samples to keep runtime manageable on CPU
DATA_DIR = pathlib.Path('data')

np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {bool(tf.config.list_physical_devices('GPU'))}")

/Users/v1neet3/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TensorFlow : 2.20.0
GPU        : False


## 2. Load & Explore the Data

In [2]:
df = pd.read_csv(DATA_DIR / 'styles.csv', on_bad_lines='skip')

# Add image path and verify the file exists
df['image_path'] = df['id'].apply(lambda x: str(DATA_DIR / 'images' / f'{x}.jpg'))
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

print(f"Total valid samples : {len(df)}")
print(f"Columns             : {df.columns.tolist()}")
df.head(3)

Total valid samples : 44419
Columns             : ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image_path']


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_path
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,data/images/15970.jpg
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,data/images/39386.jpg
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch,data/images/59263.jpg


In [3]:
# Use subCategory as target — good balance between too broad (5 classes)
# and too granular (143 article types with severe class imbalance)
TARGET = 'subCategory'

# Keep classes with >= 200 samples for reliable training
counts   = df[TARGET].value_counts()
keep     = counts[counts >= 200].index
df       = df[df[TARGET].isin(keep)].reset_index(drop=True)

CLASS_NAMES = sorted(df[TARGET].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Samples after filter: {len(df)}")

Classes (22): ['Bags', 'Belts', 'Bottomwear', 'Dress', 'Eyewear', 'Flip Flops', 'Fragrance', 'Headwear', 'Innerwear', 'Jewellery', 'Lips', 'Loungewear and Nightwear', 'Makeup', 'Nails', 'Sandal', 'Saree', 'Shoes', 'Socks', 'Ties', 'Topwear', 'Wallets', 'Watches']
Samples after filter: 43409


In [4]:
counts_filtered = df[TARGET].value_counts()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(counts_filtered.index, counts_filtered.values, color='steelblue', edgecolor='black')
ax.set_title(f'Class Distribution — {TARGET}')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Min: {counts_filtered.min()} | Max: {counts_filtered.max()}")

Min: 258 | Max: 15398


/tmp/claude-501/ipykernel_1878/2994104093.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for i, cls in enumerate(CLASS_NAMES[:18]):
    ax  = axes[i // 6][i % 6]
    row = df[df[TARGET] == cls].iloc[0]
    img = mpimg.imread(row['image_path'])
    ax.imshow(img)
    ax.set_title(cls, fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Images per Class', y=1.01)
plt.tight_layout()
plt.show()

/tmp/claude-501/ipykernel_1878/374720675.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Prepare Dataset

In [6]:
le = LabelEncoder()
df['label'] = le.fit_transform(df[TARGET])

# Subsample to keep CPU training fast
df_sub = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(min(len(x), SUBSET // NUM_CLASSES), random_state=SEED)
).reset_index(drop=True)

print(f"Using {len(df_sub)} samples across {NUM_CLASSES} classes")

# Stratified split: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(df_sub, test_size=0.3, stratify=df_sub['label'], random_state=SEED)
val_df,  test_df  = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Using 9332 samples across 22 classes
Train: 6532 | Val: 1400 | Test: 1400


/tmp/claude-501/ipykernel_1878/1015808070.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sub = df.groupby('label', group_keys=False).apply(


In [7]:
# Compute class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_df['label'].values
)
class_weight_dict = dict(enumerate(class_weights))

def make_dataset(dataframe, training=False):
    paths  = dataframe['image_path'].values
    labels = dataframe['label'].values

    def load_and_preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0
        if training:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.15)
            img = tf.image.random_contrast(img, 0.85, 1.15)
            img = tf.image.random_saturation(img, 0.85, 1.15)
            img = tf.image.random_hue(img, 0.05)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

print(f"Train batches: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Train batches: 103 | Val: 22 | Test: 22


In [8]:
# Visualise one training batch
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i >= len(images): ax.axis('off'); continue
    ax.imshow(images[i].numpy())
    ax.set_title(CLASS_NAMES[labels[i].numpy()], fontsize=7)
    ax.axis('off')
plt.suptitle('Sample Training Batch (with augmentation)', y=1.01)
plt.tight_layout()
plt.show()

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.21087968..1.078733].


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.20292285..1.0823838].


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..1.011891].


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.76801175..1.0957202].


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.17074402..1.0540038].


...

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.2496607..1.0557647].


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.07361272..0.94172263].


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..1.0417254].


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.08333273..0.9084857].


/tmp/claude-501/ipykernel_1878/2263637744.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Model 1 — MobileNetV2 Transfer Learning

Fast, lightweight, good for mobile/edge deployment.

In [9]:
def build_mobilenet(num_classes):
    base = applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = applications.mobilenet_v2.preprocess_input(inp * 255.0)
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inp, out, name='MobileNetV2')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model, base

mobilenet, mobilenet_base = build_mobilenet(NUM_CLASSES)
mobilenet.summary()

Model: "MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 22)             │         5,654 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,596,694 (9.91 MB)

 Trainable params: 336,150 (1.28 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [10]:
history_mn = mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=1, monitor='val_loss'),
    ],
    verbose=2,
)

Epoch 1/5


103/103 - 8s - 81ms/step - accuracy: 0.7593 - loss: 0.8265 - val_accuracy: 0.8643 - val_loss: 0.3960 - learning_rate: 0.0010


Epoch 2/5


103/103 - 6s - 58ms/step - accuracy: 0.8833 - loss: 0.3448 - val_accuracy: 0.8943 - val_loss: 0.3116 - learning_rate: 0.0010


Epoch 3/5


103/103 - 6s - 62ms/step - accuracy: 0.9081 - loss: 0.2617 - val_accuracy: 0.9036 - val_loss: 0.2921 - learning_rate: 0.0010


Epoch 4/5


103/103 - 6s - 63ms/step - accuracy: 0.9160 - loss: 0.2249 - val_accuracy: 0.9050 - val_loss: 0.3060 - learning_rate: 0.0010


Epoch 5/5


103/103 - 7s - 67ms/step - accuracy: 0.9372 - loss: 0.1753 - val_accuracy: 0.9086 - val_loss: 0.2854 - learning_rate: 5.0000e-04


In [11]:
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-20]:
    layer.trainable = False

mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_mn_ft = mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'),
    ],
    verbose=2,
)

Epoch 1/3


103/103 - 13s - 124ms/step - accuracy: 0.8547 - loss: 0.4376 - val_accuracy: 0.8957 - val_loss: 0.3274


Epoch 2/3


103/103 - 9s - 89ms/step - accuracy: 0.8863 - loss: 0.3508 - val_accuracy: 0.8936 - val_loss: 0.3473


Epoch 3/3


103/103 - 10s - 93ms/step - accuracy: 0.8983 - loss: 0.2970 - val_accuracy: 0.8929 - val_loss: 0.3578


## 5. Model 2 — EfficientNetV2-S Transfer Learning

Higher accuracy than MobileNetV2 at similar speed. Better suited for server-side inference.

In [12]:
def build_efficientnet(num_classes):
    base = applications.EfficientNetV2S(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = applications.efficientnet_v2.preprocess_input(inp * 255.0)
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(512, activation='relu')(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inp, out, name='EfficientNetV2S')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model, base

efficientnet, efficientnet_base = build_efficientnet(NUM_CLASSES)
efficientnet.summary()

Model: "EfficientNetV2S"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply_1 (Multiply)           │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 3, 3, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 22)             │        11,286 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,003,638 (80.12 MB)

 Trainable params: 669,718 (2.55 MB)

 Non-trainable params: 20,333,920 (77.57 MB)

In [13]:
history_en = efficientnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=1, monitor='val_loss'),
    ],
    verbose=2,
)

Epoch 1/3


103/103 - 52s - 507ms/step - accuracy: 0.7281 - loss: 0.9704 - val_accuracy: 0.8271 - val_loss: 0.6074 - learning_rate: 0.0010


Epoch 2/3


103/103 - 45s - 436ms/step - accuracy: 0.8305 - loss: 0.5454 - val_accuracy: 0.8807 - val_loss: 0.3730 - learning_rate: 0.0010


Epoch 3/3


103/103 - 42s - 412ms/step - accuracy: 0.8520 - loss: 0.4513 - val_accuracy: 0.8900 - val_loss: 0.3279 - learning_rate: 0.0010


In [14]:
efficientnet_base.trainable = True
for layer in efficientnet_base.layers[:-30]:
    layer.trainable = False

efficientnet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_en_ft = efficientnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'),
    ],
    verbose=2,
)

Epoch 1/2


103/103 - 60s - 583ms/step - accuracy: 0.8420 - loss: 0.5052 - val_accuracy: 0.8921 - val_loss: 0.3468


Epoch 2/2


103/103 - 44s - 427ms/step - accuracy: 0.8460 - loss: 0.4591 - val_accuracy: 0.8950 - val_loss: 0.3362


## 6. Evaluate & Compare

In [15]:
mn_loss, mn_acc = mobilenet.evaluate(test_ds, verbose=0)
en_loss, en_acc = efficientnet.evaluate(test_ds, verbose=0)

results = pd.DataFrame({
    'Model':      ['MobileNetV2', 'EfficientNetV2-S'],
    'Test Acc':   [mn_acc,  en_acc],
    'Test Loss':  [mn_loss, en_loss],
    'Params':     [mobilenet.count_params(), efficientnet.count_params()],
})
print(results.to_string(index=False))

           Model  Test Acc  Test Loss   Params
     MobileNetV2  0.900000   0.294522  2596694
EfficientNetV2-S  0.901429   0.328342 21003638


In [16]:
def merge_histories(*hists):
    merged = {}
    for h in hists:
        for k, v in h.history.items():
            merged.setdefault(k, []).extend(v)
    return merged

mn_hist = merge_histories(history_mn, history_mn_ft)
en_hist = merge_histories(history_en, history_en_ft)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for metric, ax in zip(['accuracy', 'loss'], axes):
    ax.plot(mn_hist[metric],           label='MobileNetV2 — train')
    ax.plot(mn_hist[f'val_{metric}'],  label='MobileNetV2 — val')
    ax.plot(en_hist[metric],           label='EfficientNetV2-S — train', ls='--')
    ax.plot(en_hist[f'val_{metric}'],  label='EfficientNetV2-S — val',   ls='--')
    ax.set_title(metric.capitalize())
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

/tmp/claude-501/ipykernel_1878/3839307645.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
best_model = efficientnet if en_acc >= mn_acc else mobilenet
best_name  = 'EfficientNetV2-S' if en_acc >= mn_acc else 'MobileNetV2'

y_true, y_pred = [], []
for imgs, lbls in test_ds:
    preds  = best_model.predict(imgs, verbose=0).argmax(axis=1)
    y_pred.extend(preds)
    y_true.extend(lbls.numpy())

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

2026-06-04 01:40:38.079852: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/tmp/claude-501/ipykernel_1878/2238378317.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
pd.DataFrame(report).T.round(3)

,precision,recall,f1-score,support
Bags,0.892,0.853,0.872,68.000
Belts,0.985,0.985,0.985,68.000
Bottomwear,0.818,0.926,0.869,68.000
Dress,0.803,0.826,0.814,69.000
Eyewear,1.000,1.000,1.000,68.000
Flip Flops,0.875,0.926,0.900,68.000
Fragrance,0.912,0.912,0.912,68.000
Headwear,0.913,0.955,0.933,44.000
Innerwear,0.938,0.882,0.909,68.000
Jewellery,0.942,0.956,0.949,68.000


## 7. Inference on Real Product Images

In [19]:
def predict_and_show(model, df_subset, n=10):
    sample = df_subset.sample(n, random_state=SEED)
    paths  = sample['image_path'].values
    true_labels = sample['label'].values

    imgs = np.stack([
        tf.image.resize(tf.cast(tf.image.decode_jpeg(
            tf.io.read_file(p), channels=3), tf.float32) / 255.0,
            [IMG_SIZE, IMG_SIZE]).numpy()
        for p in paths
    ])
    preds = model.predict(imgs, verbose=0)

    fig, axes = plt.subplots(2, 5, figsize=(16, 7))
    for i, ax in enumerate(axes.flat):
        pred_idx   = preds[i].argmax()
        confidence = preds[i][pred_idx]
        predicted  = CLASS_NAMES[pred_idx]
        actual     = CLASS_NAMES[true_labels[i]]
        color      = 'green' if pred_idx == true_labels[i] else 'red'
        ax.imshow(imgs[i])
        ax.set_title(f'Pred: {predicted} ({confidence:.0%})\nTrue: {actual}',
                     color=color, fontsize=8)
        ax.axis('off')
    plt.suptitle(f'Real Product Predictions — {best_name}', y=1.01)
    plt.tight_layout()
    plt.show()

predict_and_show(best_model, test_df)

/tmp/claude-501/ipykernel_1878/2356774155.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Multi-Label Head — Add Colour Prediction

Real e-commerce needs colour tags too. We add a second head to predict `baseColour` from the same backbone — no extra inference cost.

In [20]:
colour_counts = df['baseColour'].value_counts()
top_colours   = colour_counts[colour_counts >= 200].index.tolist()
df_multi      = df_sub[df_sub['baseColour'].isin(top_colours)].copy().reset_index(drop=True)

le_colour = LabelEncoder()
df_multi['colour_label'] = le_colour.fit_transform(df_multi['baseColour'])
NUM_COLOURS  = len(le_colour.classes_)
COLOUR_NAMES = le_colour.classes_.tolist()

print(f"Colour classes ({NUM_COLOURS}): {COLOUR_NAMES}")
print(f"Multi-label samples: {len(df_multi)}")

Colour classes (21): ['Beige', 'Black', 'Blue', 'Brown', 'Charcoal', 'Cream', 'Gold', 'Green', 'Grey', 'Maroon', 'Multi', 'Navy Blue', 'Olive', 'Orange', 'Pink', 'Purple', 'Red', 'Silver', 'Steel', 'White', 'Yellow']
Multi-label samples: 8662


In [21]:
train_m, temp_m = train_test_split(df_multi, test_size=0.3, stratify=df_multi['label'], random_state=SEED)
val_m,  test_m  = train_test_split(temp_m,   test_size=0.5, stratify=temp_m['label'],  random_state=SEED)

def make_multi_dataset(dataframe, training=False):
    paths   = dataframe['image_path'].values
    cat_lbl = dataframe['label'].values
    col_lbl = dataframe['colour_label'].values

    def load(path, cat, col):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0
        if training:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.15)
            img = tf.image.random_contrast(img, 0.85, 1.15)
        return img, {'category': cat, 'colour': col}

    ds = tf.data.Dataset.from_tensor_slices((paths, cat_lbl, col_lbl))
    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

train_mds = make_multi_dataset(train_m, training=True)
val_mds   = make_multi_dataset(val_m)
test_mds  = make_multi_dataset(test_m)

In [22]:
def build_multi_head(num_classes, num_colours):
    base = applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = applications.mobilenet_v2.preprocess_input(inp * 255.0)
    x   = base(x, training=False)
    emb = layers.GlobalAveragePooling2D(name='embedding')(x)

    cat = layers.BatchNormalization()(emb)
    cat = layers.Dropout(0.3)(cat)
    cat = layers.Dense(256, activation='relu')(cat)
    cat_out = layers.Dense(num_classes, activation='softmax', name='category')(cat)

    col = layers.Dense(128, activation='relu')(emb)
    col_out = layers.Dense(num_colours, activation='softmax', name='colour')(col)

    model = models.Model(inp, [cat_out, col_out], name='MultiHead_MobileNet')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss={'category': 'sparse_categorical_crossentropy',
              'colour':   'sparse_categorical_crossentropy'},
        loss_weights={'category': 1.0, 'colour': 0.5},
        metrics={'category': 'accuracy', 'colour': 'accuracy'},
    )
    return model

multi_model = build_multi_head(NUM_CLASSES, NUM_COLOURS)
multi_model.summary()

history_multi = multi_model.fit(
    train_mds,
    validation_data=val_mds,
    epochs=4,
    callbacks=[
        callbacks.EarlyStopping(
            patience=2, restore_best_weights=True,
            monitor='val_category_accuracy', mode='max'),
    ],
    verbose=2,
)

results_multi = multi_model.evaluate(test_mds, verbose=0)
for name, val in zip(multi_model.metrics_names, results_multi):
    print(f'{name:40s}: {val:.4f}')

Model: "MultiHead_MobileNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 96, 96, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_2          │ (None, 96, 96, 3) │          0 │ input_layer_5[0]… │
│ (Multiply)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ true_divide_1       │ (None, 96, 96, 3) │          0 │ multiply_2[0][0]  │
│ (TrueDivide)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract_1          │ (None, 96, 96, 3) │          0 │ true_divide_1[0]… │
│ (Subtract)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_96 │ (None, 3, 3,      │  2,257,984 │ subtract_1[0][0]  │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1280)      │      5,120 │ embedding[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 1280)      │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │    327,936 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │    163,968 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category (Dense)    │ (None, 22)        │      5,654 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ colour (Dense)      │ (None, 21)        │      2,709 │ dense_5[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,763,371 (10.54 MB)

 Trainable params: 502,827 (1.92 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

...

Epoch 3/4


95/95 - 10s - 102ms/step - category_accuracy: 0.9362 - category_loss: 0.1824 - colour_accuracy: 0.5855 - colour_loss: 1.3042 - loss: 0.8347 - val_category_accuracy: 0.9069 - val_category_loss: 0.3019 - val_colour_accuracy: 0.5312 - val_colour_loss: 1.5036 - val_loss: 1.0618


Epoch 4/4


95/95 - 9s - 90ms/step - category_accuracy: 0.9510 - category_loss: 0.1393 - colour_accuracy: 0.6220 - colour_loss: 1.1670 - loss: 0.7221 - val_category_accuracy: 0.9061 - val_category_loss: 0.3245 - val_colour_accuracy: 0.5343 - val_colour_loss: 1.4934 - val_loss: 1.0807


loss                                    : 1.0346
compile_metrics                         : 0.2434
category_loss                           : 1.5848
colour_loss                             : 0.9177


In [23]:
sample = test_m.sample(6, random_state=SEED)
imgs = np.stack([
    tf.image.resize(tf.cast(tf.image.decode_jpeg(
        tf.io.read_file(p), channels=3), tf.float32) / 255.0,
        [IMG_SIZE, IMG_SIZE]).numpy()
    for p in sample['image_path']
])
cat_preds, col_preds = multi_model.predict(imgs, verbose=0)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    cat  = CLASS_NAMES[cat_preds[i].argmax()]
    col  = COLOUR_NAMES[col_preds[i].argmax()]
    true_cat = CLASS_NAMES[sample.iloc[i]['label']]
    true_col = sample.iloc[i]['baseColour']
    ax.set_title(f'Pred: {cat} | {col}\nTrue: {true_cat} | {true_col}', fontsize=8)
    ax.axis('off')
plt.suptitle('Multi-Head: Category + Colour Predictions', y=1.01)
plt.tight_layout()
plt.show()

/tmp/claude-501/ipykernel_1878/2374574966.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Hierarchical Label Fallback

In [24]:
# Build subCategory → masterCategory mapping from the dataset
COARSE_MAP = df.groupby('subCategory')['masterCategory'].agg(lambda x: x.mode()[0]).to_dict()
CONFIDENCE_THRESHOLD = 0.60

def predict_with_fallback(model, images, threshold=CONFIDENCE_THRESHOLD):
    probs = model.predict(images, verbose=0)
    # Handle multi-output models
    if isinstance(probs, list):
        probs = probs[0]
    results = []
    for p in probs:
        best_idx   = p.argmax()
        best_conf  = float(p[best_idx])
        fine_label = CLASS_NAMES[best_idx]
        if best_conf >= threshold:
            results.append({'label': fine_label, 'confidence': best_conf, 'fallback': False})
        else:
            coarse = COARSE_MAP.get(fine_label, 'Unknown')
            results.append({'label': coarse, 'confidence': best_conf, 'fallback': True})
    return results

# Demo
sample = test_df.sample(8, random_state=SEED)
imgs = np.stack([
    tf.image.resize(tf.cast(tf.image.decode_jpeg(
        tf.io.read_file(p), channels=3), tf.float32) / 255.0,
        [IMG_SIZE, IMG_SIZE]).numpy()
    for p in sample['image_path']
])
preds = predict_with_fallback(best_model, imgs)

fallback_df = pd.DataFrame([
    {'True': CLASS_NAMES[sample.iloc[i]['label']],
     'Prediction': r['label'],
     'Confidence': f"{r['confidence']:.1%}",
     'Used fallback': r['fallback']}
    for i, r in enumerate(preds)
])
print(fallback_df.to_string(index=False))

      True Prediction Confidence  Used fallback
Flip Flops Flip Flops      99.6%          False
     Nails      Nails     100.0%          False
     Dress      Dress      99.6%          False
    Sandal     Sandal      99.8%          False
      Bags       Bags      96.5%          False
Flip Flops Flip Flops      71.5%          False
     Saree      Saree     100.0%          False
    Makeup     Makeup     100.0%          False


## 10. Save Models

In [25]:
mobilenet.save('mobilenet_myntra.keras')
efficientnet.save('efficientnet_myntra.keras')
multi_model.save('multihead_myntra.keras')

# Save class metadata
meta = {
    'class_names':   CLASS_NAMES,
    'colour_names':  COLOUR_NAMES,
    'coarse_map':    COARSE_MAP,
    'num_classes':   NUM_CLASSES,
    'num_colours':   NUM_COLOURS,
    'img_size':      IMG_SIZE,
    'target':        TARGET,
}
with open('myntra_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved: mobilenet_myntra.keras')
print('Saved: efficientnet_myntra.keras')
print('Saved: multihead_myntra.keras')
print('Saved: myntra_meta.json')

Saved: mobilenet_myntra.keras
Saved: efficientnet_myntra.keras
Saved: multihead_myntra.keras
Saved: myntra_meta.json


## Summary

| Upgrade | Fashion-MNIST | Myntra Dataset |
|---|---|---|
| Images | 70,000 grayscale 28×28 | 44,000 RGB 224×224 |
| Classes | 10 (broad) | 20+ subCategories |
| Label richness | Category only | Category + Colour + Gender + Season |
| Real-world relevance | Low | High (actual product photos) |
| Models | Simple CNN + MobileNetV2 | MobileNetV2 + EfficientNetV2-S |
| Multi-label | Synthesised attributes | Real colour labels |

**Next steps:**
- Scale to DeepFashion (800K images) for higher accuracy
- Add gender and season as additional output heads
- Export best model to ONNX for cross-platform deployment
- Retrain the HF Spaces demo with the new model weights